# 6 · Classical-Network Effects on QKD

**The core QFabric contribution.** Pure quantum-network simulators assume an *ideal*
classical control channel. QFabric runs BB84's classical sifting over the real FABRIC
link, so we can measure how classical-network conditions affect QKD — something
SeQUeNCe/NetSquid cannot show.

Key point: because the classical channel is TCP (reliable), latency/jitter/loss do **not**
change QBER or bits-per-photon — they change **time-to-key** and effective **key bits/second**.
This notebook impairs *only* the classical channel (TCP:5100, leaving the photon data plane
untouched) across a set of conditions and plots the impact.

### At a glance
- **Purpose:** quantify how classical-channel latency/jitter/loss affect BB84 throughput.
- **Inputs:** `SLICE_NAME`, `SCENARIO`, and a list of network `conditions`.
- **Outputs:** `results/network_effects.json` + plots (bits/s, time-to-key, QBER per condition).
- **Runs on / runtime:** FABRIC JupyterHub; ~`len(conditions)` × a BB84 run (a few min each).
- **If something fails:** a severe condition may time out → that row is recorded as failed; netem is always cleared at the end (and on each iteration).

**Prerequisite:** `01_setup_slice` completed (switch running, data-plane IPs up).

## Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO   = 'validation/scenarios/fabric_1km.yml'

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## Step 1 — Run BB84 under each classical-network condition
Each condition impairs only TCP:5100 (the sifting channel); photons are untouched. Edit
the list to taste — `delay_ms`, `jitter_ms`, `loss_pct`, or asymmetric `alice_delay_ms`/`bob_delay_ms`.

In [ ]:
conditions = [
    {'name': 'baseline'},
    {'name': 'latency_25ms',     'delay_ms': 25},
    {'name': 'latency_100ms',    'delay_ms': 100},
    {'name': 'jitter_50pm20ms',  'delay_ms': 50, 'jitter_ms': 20},
    {'name': 'loss_1pct',        'loss_pct': 1.0},
    {'name': 'asymmetric_100_10','alice_delay_ms': 100, 'bob_delay_ms': 10},
]
rows = df.run_network_conditions_experiment(slice_obj, SCENARIO, conditions)
print(f'\n{len(rows)} conditions recorded -> results/network_effects.json')

## Step 2 — Plots: throughput & time-to-key degrade, QBER stays flat
Loads `results/network_effects.json` (so this also re-plots a saved run). The story:
**bits/second and time-to-key respond to the classical channel; QBER does not** — the
effect ideal-channel simulators miss.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
rows = json.loads((PROJECT_DIR / 'results' / 'network_effects.json').read_text())
ok = [r for r in rows if not r.get('error')]
names = [r['condition'] for r in ok]
bps   = [r.get('key_bits_per_sec', 0) for r in ok]
tts   = [r.get('elapsed_seconds', 0) for r in ok]
qber  = [r.get('qber', 0) * 100 for r in ok]

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].bar(names, bps, color='seagreen');   ax[0].set(ylabel='key bits / second', title='Throughput')
ax[1].bar(names, tts, color='steelblue');  ax[1].set(ylabel='seconds', title='Time to key')
ax[2].bar(names, qber, color='indianred'); ax[2].set(ylabel='QBER (%)', title='QBER (should be ~flat)')
for a in ax: a.tick_params(axis='x', rotation=35); a.grid(axis='y', alpha=0.3)
fig.suptitle('Classical-network effect on BB84 (QFabric on FABRIC)')
plt.tight_layout(); plt.show()

failed = [r['condition'] for r in rows if r.get('error')]
if failed:
    print('conditions that failed (e.g. timed out):', failed)

## Step 3 — Table

In [ ]:
import pandas as pd
pd.DataFrame(rows)[['condition','qber','sifted_bits','final_key_bits','elapsed_seconds','key_bits_per_sec']]

---
This is the headline result: QFabric measures classical-network impact on QKD throughput.
Extend `conditions` (or add competing traffic) to map the operating envelope.